In [1]:
from datasets import load_dataset
dataset = load_dataset("Davidsamuel101/ebible_local_ind_corpus", "ind_aaz")

In [2]:
dataset["train"]

Dataset({
    features: ['source_text', 'target_text', 'source_lang', 'target_lang', 'verse'],
    num_rows: 9068
})

In [2]:
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

target_langs = ["lex"]

stats = {}

length = 300
for lang in target_langs:
    if "ind_Latn" not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({"additional_special_tokens": ["ind_Latn"]})

    if f"{lang}_Latn" not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({"additional_special_tokens": [f"{lang}_Latn"]})
        
    tokenizer.src_lang = "ind_Latn"
    tokenizer.tgt_lang = f"{lang}_Latn"
    dataset = load_dataset("Davidsamuel101/ebible_local_ind_corpus", f"ind_{lang}")
    src_under_length = 0
    src_over_length = 0
    tgt_under_length = 0
    tgt_over_length = 0
    for datum in dataset["train"]:
        source_text = datum["source_text"]
        target_text = datum["target_text"]

        src_tokens = tokenizer(source_text, truncation=False, add_special_tokens=True)
        tgt_tokens = tokenizer(target_text, truncation=False, add_special_tokens=True)
        src_len = len(src_tokens['input_ids'])
        tgt_len = len(tgt_tokens['input_ids'])
        if src_len <= length:
            src_under_length += 1
        else:
            src_over_length += 1
        if tgt_len <= length:
            tgt_under_length += 1
        else:
            tgt_over_length += 1
    stats[lang] = {
        "src_under_length": src_under_length,
        "src_over_length": src_over_length,
        "tgt_under_length": tgt_under_length,
        "tgt_over_length": tgt_over_length,
        "total_examples": len(dataset["train"])
    }
# Print statistics for each language, sorted by src_under_length count
sorted_langs = sorted(target_langs, key=lambda lang: stats[lang]['tgt_over_length'])
for lang in sorted_langs:
    s = stats[lang]
    print(f"Language: {lang}")
    print(f"  Source <={length}: {s['src_under_length']} ({s['src_under_length']/s['total_examples']:.2%})")
    print(f"  Source >{length}: {s['src_over_length']} ({s['src_over_length']/s['total_examples']:.2%})")
    print(f"  Target <={length}: {s['tgt_under_length']} ({s['tgt_under_length']/s['total_examples']:.2%})")
    print(f"  Target >{length}: {s['tgt_over_length']} ({s['tgt_over_length']/s['total_examples']:.2%})")
    print(f"  Total examples: {s['total_examples']}")
    print("-" * 40)


Language: lex
  Source <=300: 9674 (100.00%)
  Source >300: 0 (0.00%)
  Target <=300: 9672 (99.98%)
  Target >300: 2 (0.02%)
  Total examples: 9674
----------------------------------------


In [3]:
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

target_langs = ["nfa"]

stats = {}

length = 300
for lang in target_langs:
    if "ind_Latn" not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({"additional_special_tokens": ["ind_Latn"]})

    if f"{lang}_Latn" not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({"additional_special_tokens": [f"{lang}_Latn"]})
        
    tokenizer.src_lang = "ind_Latn"
    tokenizer.tgt_lang = f"{lang}_Latn"
    dataset = load_dataset("Davidsamuel101/ebible_local_ind_corpus", f"ind_{lang}")
    src_under_length = 0
    src_over_length = 0
    tgt_under_length = 0
    tgt_over_length = 0
    with open("/data/projects/punim0478/setiawand/bible-nmt/ebible-corpus/dhao-eng/nfa-nfa-norm.txt") as f:
        text = f.readlines()

        src_tokens = tokenizer(text, truncation=False, add_special_tokens=True)
        tgt_tokens = tokenizer(text, truncation=False, add_special_tokens=True)
        src_len = len(src_tokens['input_ids'])
        tgt_len = len(tgt_tokens['input_ids'])
        if src_len <= length:
            src_under_length += 1
        else:
            src_over_length += 1
        if tgt_len <= length:
            tgt_under_length += 1
        else:
            tgt_over_length += 1
    stats[lang] = {
        "src_under_length": src_under_length,
        "src_over_length": src_over_length,
        "tgt_under_length": tgt_under_length,
        "tgt_over_length": tgt_over_length,
        "total_examples": len(dataset["train"])
    }
# Print statistics for each language, sorted by src_under_length count
sorted_langs = sorted(target_langs, key=lambda lang: stats[lang]['tgt_over_length'])
for lang in sorted_langs:
    s = stats[lang]
    print(f"Language: {lang}")
    print(f"  Source <={length}: {s['src_under_length']} ({s['src_under_length']/s['total_examples']:.2%})")
    print(f"  Source >{length}: {s['src_over_length']} ({s['src_over_length']/s['total_examples']:.2%})")
    print(f"  Target <={length}: {s['tgt_under_length']} ({s['tgt_under_length']/s['total_examples']:.2%})")
    print(f"  Target >{length}: {s['tgt_over_length']} ({s['tgt_over_length']/s['total_examples']:.2%})")
    print(f"  Total examples: {s['total_examples']}")
    print("-" * 40)


Language: nfa
  Source <=300: 0 (0.00%)
  Source >300: 1 (0.01%)
  Target <=300: 0 (0.00%)
  Target >300: 1 (0.01%)
  Total examples: 9070
----------------------------------------


In [1]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

NT_BOOKS = [
    "MAT",
    "MRK",
    "LUK",
    "JHN",
    "ACT",
    "ROM",
    "1CO",
    "2CO",
    "GAL",
    "EPH",
    "PHP",
    "COL",
    "1TH",
    "2TH",
    "1TI",
    "2TI",
    "TIT",
    "PHM",
    "HEB",
    "JAB",
    "JAS",
    "1PE",
    "2PE",
    "1JN",
    "2JN",
    "3JN",
    "JUD",
    "REV",
]

def load_ebible_local_ind_corpus(src_lang, tgt_lang):
    dataset = load_dataset("Davidsamuel101/ebible_local_ind_corpus", f"{src_lang}_{tgt_lang}")
    # OT books for testing, NT books for training and validation
    def is_nt_book(refs):
        if isinstance(refs, list):
            # Check if any ref belongs to NT books
            return any(ref.split()[0] in NT_BOOKS for ref in refs)
        else:
            # Single reference
            return refs.split()[0] in NT_BOOKS
    
    # The dataset is a DatasetDict, so we need to access the 'train' split
    train_data = dataset['train']
    train_ds = train_data.filter(lambda x: is_nt_book(x["verse"]))
    test_ds = train_data.filter(lambda x: not is_nt_book(x["verse"]))
    train_val_ds = train_ds.train_test_split(test_size=0.05, seed=41)
    dataset = DatasetDict({"train": train_val_ds["train"], "validation": train_val_ds["test"], "test": test_ds})
    print(dataset)
    return dataset

/data/projects/punim0478/setiawand/.cache/conda-envs/david/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_ebible_local_ind_corpus("ind", "ptu")

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang', 'verse'],
        num_rows: 7479
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang', 'verse'],
        num_rows: 394
    })
    test: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang', 'verse'],
        num_rows: 1472
    })
})


In [3]:
for datum in dataset["train"]:
    print(datum["verse"].split()[0])

MRK
ROM
JHN
LUK
JHN
2PE
REV
LUK
1CO
REV
ACT
PHP
1TH
ACT
1PE
HEB
JHN
ROM
MAT
1JN
2CO
ACT
HEB
ROM
ACT
LUK
JHN
1CO
JHN
REV
MAT
ACT
MRK
JHN
JHN
JHN
GAL
LUK
ROM
ROM
ACT
1CO
HEB
ROM
ACT
COL
1CO
LUK
EPH
ACT
ROM
ACT
MAT
REV
JHN
REV
EPH
ACT
MRK
MAT
REV
2CO
ROM
ACT
1TI
MAT
LUK
LUK
HEB
JHN
MAT
2TH
JHN
JHN
HEB
PHP
1TH
LUK
REV
JHN
1JN
ACT
2CO
ACT
MAT
JHN
JAS
LUK
MRK
ACT
JHN
JHN
1CO
ACT
ACT
2TI
ACT
HEB
LUK
PHP
MRK
ACT
MAT
ACT
1TI
2CO
2CO
1CO
ACT
REV
LUK
1CO
REV
REV
MAT
HEB
MAT
ROM
MRK
LUK
ACT
MAT
1JN
MAT
LUK
HEB
JHN
LUK
ACT
2CO
2PE
LUK
TIT
JHN
ACT
1PE
HEB
REV
MRK
JHN
ACT
JAS
ACT
1TI
JHN
ACT
HEB
ROM
LUK
ROM
MAT
1JN
1CO
HEB
MRK
PHP
1CO
2CO
JHN
LUK
ACT
MAT
ACT
JHN
HEB
ROM
ACT
JHN
MAT
PHM
JHN
LUK
ACT
ACT
JHN
LUK
LUK
2PE
1TH
ROM
LUK
ACT
LUK
HEB
MAT
JHN
1CO
PHP
2CO
JHN
ACT
ACT
MRK
LUK
GAL
LUK
ROM
2CO
LUK
MAT
REV
ACT
JHN
MAT
ROM
COL
1CO
ACT
HEB
LUK
ACT
LUK
LUK
ROM
JHN
ACT
JHN
JHN
JHN
LUK
1TI
1CO
REV
MRK
MAT
JHN
HEB
MAT
ROM
REV
JHN
2CO
ACT
JHN
1CO
HEB
JAS
2CO
2TI
LUK
LUK
EPH
ACT
JHN
JHN
JHN
JHN
1TI
2TH
JUD
